# Asteroid families with HCM (100,000-asteroid version)

**Steps:** load the file → keep main-belt asteroids → attach the Proper25 answer key → Zappalà features
→ HCM (single linkage cut at one speed) → score → dendrogram of one small family.

**Why this differs from the 100-asteroid notebook:** a full distance matrix for 100,000 asteroids
would need 80 GB of memory. HCM at one cutoff only needs the pairs that are *closer* than the cutoff,
so we find just those (Step 4). The result is exactly the same as cutting the single-linkage tree.

In [ ]:
import os, glob, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import radius_neighbors_graph
from scipy.sparse.csgraph import connected_components
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# ---- settings: change these if your files are somewhere else ----
ASTEROID_FILE = "/content/sample_data/asteroids_100k.csv"
PROPER25_ZIP  = "/content/Proper25.zip"
FAM_DIR       = "Proper25"

N_FULL_BELT = 825_292      # main-belt asteroids in the full SBDB file
CUT_FULL    = 45.0         # best HCM cutoff (m/s) on the full belt
MIN_MEMBERS = 10           # smallest group we call a family; smaller groups are noise (-1)
DENDRO_FAMILY = "7353 Kazuya"  # small family to draw as a dendrogram in Step 7

GM_SUN, AU_METERS = 1.32712440018e20, 1.495978707e11

## Step 1 · Load and keep only main-belt asteroids

Keeps a row only if JPL calls it a main-belt asteroid (`class == "MBA"`) **and** `a` is between
2.0 and 3.5 AU. Also extracts the asteroid **number** and **provisional designation** from
`full_name` (e.g. `4 Vesta (A807 FA)` → number 4, prov `A807FA`) for the answer-key join.

In [ ]:
df = pd.read_csv(ASTEROID_FILE)
keep = (df["class"] == "MBA") & df["a"].between(2.0, 3.5)
df = df[keep].reset_index(drop=True)

df["sin_i"]  = np.sin(np.radians(df["i"]))
df["number"] = pd.to_numeric(df["full_name"].str.extract(r"^\s*(\d+)")[0], errors="coerce")
df["prov"]   = df["full_name"].str.extract(r"\(([^)]+)\)")[0].str.replace(" ", "", regex=False)

N = len(df)
print(f"Kept {N:,} main-belt asteroids (removed {(~keep).sum():,})")

## Step 2 · Attach the Proper25 answer key

Unzip if needed, read every `.fam3` file, then match each member to our file by **number** (label is
all digits) or **provisional designation** (otherwise). If an asteroid is in two families, the
**smaller** family wins. Everyone else is `"background"`. The answer key is only used for grading.

In [ ]:
if not glob.glob(os.path.join(FAM_DIR, "**", "*.fam3"), recursive=True):
    with zipfile.ZipFile(PROPER25_ZIP) as z:
        z.extractall(FAM_DIR)

parts = []
for path in sorted(glob.glob(os.path.join(FAM_DIR, "**", "*.fam3"), recursive=True)):
    region, parent, name = os.path.basename(path).split(".")[0].split("_", 2)
    lines = pd.read_csv(path, sep=r"\s+", header=None, usecols=[0, 1, 2, 8],
                        names=["a_p", "e_p", "sini_p", "label"], dtype={"label": str})
    lines["family"], lines["region"], lines["family_size"] = f"{parent} {name}", region, len(lines)
    parts.append(lines)
members = pd.concat(parts, ignore_index=True)

is_num = members["label"].str.fullmatch(r"\d+")
matched = pd.concat([
    df[["number"]].dropna().astype(int).reset_index()
      .merge(members[is_num].assign(number=lambda m: m["label"].astype(int)), on="number"),
    df[["prov"]].dropna().reset_index()
      .merge(members[~is_num].rename(columns={"label": "prov"}), on="prov"),
])
best = matched.sort_values("family_size").drop_duplicates("index").set_index("index")
df["true_family"] = best["family"].reindex(df.index).fillna("background")
for c in ["a_p", "e_p", "sini_p"]:                       # proper elements (members only)
    df[c] = best[c].reindex(df.index)

print(f"{members['family'].nunique()} families in Proper25")
print(f"Family members in our file: {(df['true_family'] != 'background').sum():,} of {N:,}")
df["true_family"].value_counts().head(8)

## Step 3 · Zappalà features (m/s)

To use fast neighbor search we turn each orbit into three numbers so that the ordinary straight-line
distance between two asteroids **equals** the Zappalà distance:

f₁ = v·√1.25·ln a,  f₂ = v·√2·e,  f₃ = v·√2·sin i,  with v = √(GM/a_ref) at one fixed a_ref (the median a).

The only difference from the pair formula is that v uses the median a instead of each pair's average a.
The check below shows the difference is small (about 5%): a 93 m/s link might really be 88 or 98 m/s.

In [ ]:
A_REF = df["a"].median()
v = np.sqrt(GM_SUN / (A_REF * AU_METERS))
X = np.column_stack([v * np.sqrt(1.25) * np.log(df["a"]),
                     v * np.sqrt(2) * df["e"],
                     v * np.sqrt(2) * df["sin_i"]])

def zappala_distance(a1, e1, s1, a2, e2, s2):          # the pair formula, for checking
    a_mean = 0.5 * (a1 + a2)
    vv = np.sqrt(GM_SUN / (a_mean * AU_METERS))
    return vv * np.sqrt(1.25 * ((a1 - a2) / a_mean) ** 2 + 2 * (e1 - e2) ** 2 + 2 * (s1 - s2) ** 2)

# check: each of 1,000 random asteroids against its nearest neighbor (the pairs HCM cares about)
from sklearn.neighbors import NearestNeighbors
i = np.random.default_rng(0).choice(N, 1000, replace=False)
j = NearestNeighbors(n_neighbors=2).fit(X).kneighbors(X[i])[1][:, 1]
cols = ["a", "e", "sin_i"]
d_pair = zappala_distance(*(df[c].to_numpy()[i] for c in cols), *(df[c].to_numpy()[j] for c in cols))
d_feat = np.linalg.norm(X[i] - X[j], axis=1)
ok = d_pair > 0
print(f"a_ref = {A_REF:.3f} AU, v = {v:,.0f} m/s")
print(f"Median difference between feature distance and pair formula: {np.median(np.abs(d_feat[ok] / d_pair[ok] - 1)):.2%}")

## Step 4 · HCM: link close pairs, follow the chains

1. `radius_neighbors_graph` finds every pair closer than the cutoff (and only those).
2. `connected_components` follows the links: friends of friends end up in one group.
3. Groups smaller than `MIN_MEMBERS` become noise (-1).

**Cutoff:** 45 m/s is right for the full belt. Our file holds fewer asteroids, so neighbors are farther
apart; the gap grows as (N_full / N)^(1/3). For about 92,000 main-belt asteroids that is about 2×, so
the cutoff is about 94 m/s.

In [ ]:
def hcm_labels(features, d_cut, min_members=MIN_MEMBERS):
    links = radius_neighbors_graph(features, radius=d_cut, mode="connectivity", n_jobs=-1)
    _, group = connected_components(links, directed=False)
    size = np.bincount(group)
    return np.where(size[group] >= min_members, group, -1)

CUTOFF = CUT_FULL * (N_FULL_BELT / N) ** (1 / 3)
df["hcm_cluster"] = hcm_labels(X, CUTOFF)

clustered = df["hcm_cluster"] >= 0
print(f"Cutoff: {CUTOFF:.0f} m/s")
print(f"Groups with >= {MIN_MEMBERS} members: {df.loc[clustered, 'hcm_cluster'].nunique()}")
print(f"Clustered: {clustered.mean():.1%}  |  largest group: {df['hcm_cluster'][clustered].value_counts().max() / N:.1%} of the file")
df.to_csv("09_clustering_input.csv", index=False)

## Step 5 · Scoring

**Recall** = share of a family's members in its best cluster. **Purity** = share of that cluster that
belongs to the family. **Found** = both ≥ 50%. Families with fewer than `MIN_MEMBERS` members in the
file are skipped (too few to form a group). **ARI**: 1 = perfect, 0 = random; background and noise
each count as one label.

In [ ]:
NOISE, BACKGROUND = -1, "background"

def family_scores(truth, clusters):
    t = pd.DataFrame({"family": np.asarray(truth), "cluster": np.asarray(clusters)})
    cluster_size = t.loc[t["cluster"] != NOISE, "cluster"].value_counts()
    rows = []
    for fam, m in t[t["family"] != BACKGROUND].groupby("family"):
        if len(m) < MIN_MEMBERS:
            continue
        in_clusters = m.loc[m["cluster"] != NOISE, "cluster"].value_counts()
        if in_clusters.empty:
            best, n_best, recall, purity = NOISE, 0, 0.0, 0.0
        else:
            best, n_best = in_clusters.idxmax(), in_clusters.max()
            recall, purity = n_best / len(m), n_best / cluster_size[best]
        match = 0.0 if recall + purity == 0 else 2 * recall * purity / (recall + purity)
        rows.append({"family": fam, "size": len(m), "best_cluster": best, "n_in_best": n_best,
                     "recall": recall, "purity": purity, "match": match,
                     "found": recall >= 0.5 and purity >= 0.5})
    cols = ["family", "size", "best_cluster", "n_in_best", "recall", "purity", "match", "found"]
    return pd.DataFrame(rows, columns=cols).sort_values("size", ascending=False, ignore_index=True)

def score(truth, clusters):
    truth, clusters = np.asarray(truth), np.asarray(clusters)
    fam = family_scores(truth, clusters)
    clustered = clusters != NOISE
    sizes = pd.Series(clusters[clustered]).value_counts()
    return {
        "pct_clustered": 100 * clustered.mean(),
        "largest_cluster_pct": 100 * sizes.max() / len(clusters) if len(sizes) else 0.0,
        "families_scored": len(fam),
        "families_found": int(fam["found"].sum()),
        "ARI": adjusted_rand_score(truth, clusters),
        "NMI": normalized_mutual_info_score(truth, clusters),
    }, fam

summary, fam = score(df["true_family"], df["hcm_cluster"])
fam.round(3).to_csv("family_scores_hcm.csv", index=False)
print({k: round(float(v), 3) for k, v in summary.items()})
print("\nFamilies found:")
print(fam[fam["found"]].round(2).to_string(index=False))

## Step 6 · Does the cutoff matter? (families vs cutoff)

Run HCM at several cutoffs and count the groups. Too small: tiny fragments (noise). Too big: chains
merge everything. The count peaks in between, and the peak can be found **without** the answer key.

In [ ]:
rows = []
for f in [0.6, 0.8, 0.9, 1.0, 1.1, 1.2, 1.4]:
    lab = hcm_labels(X, CUTOFF * f)
    rows.append({"cutoff m/s": round(CUTOFF * f), "groups": len(set(lab) - {-1}),
                 "ARI": round(adjusted_rand_score(df["true_family"], lab), 3)})
sweep = pd.DataFrame(rows)
display(sweep)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sweep["cutoff m/s"], sweep["groups"], "o-")
ax.set(xlabel="Cutoff (m/s)", ylabel=f"Groups with >= {MIN_MEMBERS} members", title="Families vs cutoff")
plt.show()

## Step 7 · Dendrogram of one small family

A dendrogram of all 100,000 asteroids is impossible (and unreadable), so we draw one **small family**:

- **Filter:** keep every asteroid inside a box around the family's members (from their lowest to
  highest a, e and sin i, widened by a quarter of the box size on each side).
- **Top:** single-linkage tree of everything in the box, using **current orbits** (Step 3 features).
  **Orange ticks** under the leaves = catalog members of the family; leaves without a tick are other asteroids.
- **Bottom:** the same family members only, using their **proper elements** (long-term averaged orbits
  from Proper25, same m/s scale).
- The red line is the cutoff from Step 4. A family is found when its orange leaves join **below** the line.

In [ ]:
fam_rows = df[df["true_family"] == DENDRO_FAMILY]
lo, hi = fam_rows[["a", "e", "sin_i"]].min(), fam_rows[["a", "e", "sin_i"]].max()
pad = (hi - lo) / 4
in_box = np.all([df[c].between(lo[c] - pad[c], hi[c] + pad[c]) for c in ["a", "e", "sin_i"]], axis=0)
box = df[in_box]
print(f"{DENDRO_FAMILY}: {len(fam_rows)} members in the file, {len(box)} asteroids in the box")

proper = np.column_stack([v * np.sqrt(1.25) * np.log(fam_rows["a_p"]),
                          v * np.sqrt(2) * fam_rows["e_p"],
                          v * np.sqrt(2) * fam_rows["sini_p"]])

fig, axes = plt.subplots(2, 1, figsize=(15, 10))
panels = [(axes[0], X[in_box], box, "current orbits, family + neighbors"),
          (axes[1], proper, fam_rows, "proper elements, family members only")]
for ax, feats, rows, title in panels:
    tree = linkage(feats, method="single")
    leaves = dendrogram(tree, no_labels=True, color_threshold=0,
                        above_threshold_color="#7a7a7a", ax=ax)["leaves"]
    is_member = (rows["true_family"] == DENDRO_FAMILY).to_numpy()[leaves]
    x = 5 + 10 * np.arange(len(leaves))                     # leaf positions used by dendrogram()
    top = max(ax.get_ylim()[1], CUTOFF * 1.3)
    ax.vlines(x[is_member], -0.06 * top, -0.01 * top, color="#e07a2e", lw=2, label="family member")
    ax.set_ylim(-0.07 * top, top)
    ax.axhline(CUTOFF, color="red", ls="--", lw=1, label=f"cutoff {CUTOFF:.0f} m/s")
    ax.set(title=f"{DENDRO_FAMILY}: {title} ({len(rows)} asteroids, {is_member.sum()} members)",
           ylabel="Zappalà distance (m/s)")
    ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print(fam[fam["family"] == DENDRO_FAMILY].round(2).to_string(index=False))